# Progetto Computer Vision: Rilevamento Strike Zone Dinamica

Questo notebook orchestra le fasi del progetto per rilevare la home plate e la strike zone dinamica basata sulla pose estimation del battitore, utilizzando YOLOv8.

---

## 1. Setup Iniziale e Installazione Dipendenze
Assicurati di essere nel tuo ambiente `vision310` (`pyenv activate vision310`).
Installare le librerie necessarie.

In [13]:
!pip install ultralytics opencv-python pandas ipykernel


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


## 2. Preparazione dei Dati
Queste celle eseguono gli script di preparazione dei dati che hai già creato.
**Assicurati che `ohtani_strike.mp4` si trovi nella cartella `videos/` del tuo progetto.**

### 2.1 Estrazione dei Frame dal Video
Esegue lo script `extract_plate_frames.py` per estrarre i frame dal video `ohtani_strike.mp4`.
I frame verranno salvati in `annotations/plate/images/`.

In [31]:
import subprocess
import os

print("Esecuzione di extract_plate_frames.py...")
# Assicurati che il percorso sia corretto per il tuo progetto
path_to_script = "/Users/mattiagugole/Desktop/vision_project/utils/extract_plate_frames.py"
subprocess.run(["python", path_to_script], check=True)
print("Estrazione frame completata.")

Esecuzione di extract_plate_frames.py...
Extracted 70 frames to: /Users/mattiagugole/Desktop/vision_project/annotations/plate/images
Estrazione frame completata.


### 2.2 Promemoria per l'Annotazione e Consolidamento del Dataset (Manuale - Già Fatto!)

Questo passaggio cruciale è stato completato manualmente:
- Hai annotato la `home plate` e la `bat` nei frame utilizzando Label Studio.
- Hai consolidato tutte le immagini e i file `.txt` delle annotazioni (sia per 'plate' che per 'bat') in una singola cartella:
  `/Users/mattiagugole/Desktop/vision_project/annotations/bat_n_plate/`
  - Le immagini sono in `images/`
  - I file `.txt` sono in `labels/` e contengono le etichette per entrambe le classi (`0` per plate, `1` per bat).

**Assicurati che le cartelle originali `annotations/plate/`, `annotations/plate_detection/` e `datasets/plate_detection/` siano state rimosse per pulizia, come discusso in precedenza.**

### 2.3 Pulizia Nomi File delle Label Combinate

Esegue il nuovo script `rename_combined_labels.py` per correggere i nomi dei file `.txt` generati da Label Studio (con i nomi lunghi e hash), rendendoli conformi al formato `frame_XXXX.txt` nella cartella `bat_n_plate/labels/`.

In [98]:
import subprocess

print("Esecuzione di rename_combined_labels.py...")
# Assicurati che il percorso sia corretto per il tuo script in utils/
path_to_script = "/Users/mattiagugole/Desktop/vision_project/utils/rename_combined_labels.py"
subprocess.run(["python", path_to_script], check=True)
print("Pulizia nomi file label combinati completata.")

Esecuzione di rename_combined_labels.py...
Inizio pulizia nomi file nella cartella: /Users/mattiagugole/Desktop/vision_project/annotations/bat_n_plate_n_ball/labels
Skipped: 99bb04da-frame_0170.txt (new name 'frame_0170.txt' does not match expected frame_XXXX.txt format after split)
Skipped: 69047034-frame_0290.txt (new name 'frame_0290.txt' does not match expected frame_XXXX.txt format after split)
Skipped: 41e443a3-frame_0270.txt (new name 'frame_0270.txt' does not match expected frame_XXXX.txt format after split)
Skipped: 5205433d-frame_0045.txt (new name 'frame_0045.txt' does not match expected frame_XXXX.txt format after split)
Skipped: da7033f2-frame_0325.txt (new name 'frame_0325.txt' does not match expected frame_XXXX.txt format after split)
Skipped: 3306b3e9-frame_0315.txt (new name 'frame_0315.txt' does not match expected frame_XXXX.txt format after split)
Skipped: 69f47e66-frame_0175.txt (new name 'frame_0175.txt' does not match expected frame_XXXX.txt format after split)
Sk

### 2.4 Suddivisione del Dataset (Train/Validation)
Esegue lo script `split_dataset.py` per suddividere le immagini e le label in set di training (80%) e validation (20%).
I dati verranno copiati in `datasets/plate_detection/{images,labels}/{train,val}`.
Verrà anche eseguito `remove.py` per pulire cartelle temporanee/errate.

In [99]:
import subprocess
import os

print("Esecuzione di split_dataset.py per il dataset unificato...")
# Assicurati che il percorso sia corretto per il tuo script split_dataset.py
path_to_split_script = "/Users/mattiagugole/Desktop/vision_project/utils/split_dataset.py"
subprocess.run(["python", path_to_split_script], check=True)
print("Suddivisione dataset unificato completata.")

# Il tuo vecchio script remove.py non è più necessario qui, in quanto split_dataset.py
# si occupa della creazione delle cartelle target.
# Se lo script remove.py originale è ancora presente, assicurati che i suoi percorsi
# non interferiscano con la nuova struttura. Era progettato per plate_detection.

Esecuzione di split_dataset.py per il dataset unificato...
Cartelle di output create in: /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection
Totale immagini: 71
Immagini per il Training: 56
Immagini per la Validation: 15

Copia dei file di training...
Copia dei file di validation...

Dataset suddiviso e copiato correttamente.
I dati sono ora disponibili in: /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection/images/train, /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection/images/val, etc.
Suddivisione dataset unificato completata.


### 2.5 Verifica `data_combined.yaml`

Assicurati che il tuo nuovo file `data_combined.yaml` (nella cartella `annotations/bat_n_plate/`) sia configurato correttamente per **entrambe le classi `plate` e `bat`** e punti alle nuove cartelle `datasets/combined_detection/`.

**Contenuto atteso di `data_combined.yaml`:**
```yaml
train: /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection/images/train
val: /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection/images/val

nc: 2
names: ['plate', 'bat']

In [107]:
from ultralytics import YOLO
import os
import shutil

print("--- Preparazione per l'addestramento del Modello YOLOv8 Unificato (Plate + Bat) ---")

# --- Controllo e Pulizia di Runs Precedenti ---
runs_detect_path = "/Users/mattiagugole/Desktop/vision_project/codice/runs/detect/"
combined_detector_base_name = "combined_detector" 
existing_runs = [d for d in os.listdir(runs_detect_path) if os.path.isdir(os.path.join(runs_detect_path, d)) and d.startswith(combined_detector_base_name)]

if len(existing_runs) > 0: 
    print(f"⚠️ ATTENZIONE: Trovate cartelle di addestramento esistenti per '{combined_detector_base_name}': {existing_runs}")
    print("   Il prossimo addestramento creerà una nuova cartella come 'combined_detectorX'.")
    print("   Ricorda di aggiornare il percorso del modello nel file 'utils/extract_data_for_strike_zone.py' di conseguenza.")
else:
    print(f"✅ Nessuna cartella di addestramento precedente trovata per '{combined_detector_base_name}'. Verrà creata 'combined_detector'.")

print("\nInizio addestramento YOLOv8 per Plate e Bat (Modello Unificato)...")
model_combined_detector = YOLO('yolov8n.pt') 

# Addestra il modello sul dataset unificato (plate + bat)
results_combined = model_combined_detector.train(
    data='/Users/mattiagugole/Desktop/vision_project/annotations/bat_n_plate_n_ball/data_combined.yaml', # <--- PERCORSO AGGIORNATO QUI!
    epochs=100, 
    imgsz=640, 
    device='cpu', 
    name=combined_detector_base_name 
)

# Dopo l'addestramento, identifica il nome della cartella creata
created_combined_runs = [d for d in os.listdir(runs_detect_path) if os.path.isdir(os.path.join(runs_detect_path, d)) and d.startswith(combined_detector_base_name)]
last_combined_run_folder = sorted(created_combined_runs, key=lambda x: int(x.replace(combined_detector_base_name, '') or 0))[-1]

print(f"\nAddestramento combinato completato. Modello salvato in: {runs_detect_path}{last_combined_run_folder}/weights/best.pt")
print("⭐⭐⭐ RICORDA: Aggiorna il percorso del modello `model_combined_detector_path` nel file `utils/extract_data_for_strike_zone.py`")
print(f"         con: '{runs_detect_path}{last_combined_run_folder}/weights/best.pt'")

--- Preparazione per l'addestramento del Modello YOLOv8 Unificato (Plate + Bat) ---
⚠️ ATTENZIONE: Trovate cartelle di addestramento esistenti per 'combined_detector': ['combined_detector']
   Il prossimo addestramento creerà una nuova cartella come 'combined_detectorX'.
   Ricorda di aggiornare il percorso del modello nel file 'utils/extract_data_for_strike_zone.py' di conseguenza.

Inizio addestramento YOLOv8 per Plate e Bat (Modello Unificato)...
New https://pypi.org/project/ultralytics/8.3.152 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.133 🚀 Python-3.10.13 torch-2.7.0 CPU (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/mattiagugole/Desktop/vision_project/annotations/bat_n_plate_n_ball/data_combined.yaml, degrees=0.0, determinist

train: Scanning /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection/labels/train.cache... 66 images, 0 backgrounds, 0 corrupt: 100%|██████████| 66/66 [00:00<?, ?it/s]

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1081.3±149.6 MB/s, size: 340.9 KB)



val: Scanning /Users/mattiagugole/Desktop/vision_project/datasets/combined_detection/labels/val.cache... 25 images, 0 backgrounds, 0 corrupt: 100%|██████████| 25/25 [00:00<?, ?it/s]

Plotting labels to runs/detect/combined_detector2/labels.jpg... 


...


    100/100         0G      1.098       1.05     0.9115          4        640: 100%|██████████| 5/5 [00:22<00:00,  4.59s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.60s/it]

                   all         25         57      0.421      0.718      0.481      0.297

100 epochs completed in 0.734 hours.
Optimizer stripped from runs/detect/combined_detector2/weights/last.pt, 6.2MB


Optimizer stripped from runs/detect/combined_detector2/weights/best.pt, 6.2MB

Validating runs/detect/combined_detector2/weights/best.pt...
Ultralytics 8.3.133 🚀 Python-3.10.13 torch-2.7.0 CPU (Apple M1 Pro)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:02<00:00,  2.50s/it]


                   all         25         57      0.371      0.627      0.516      0.326
                  ball         17         17      0.186      0.162      0.248      0.142
                   bat         25         25      0.461       0.72      0.561      0.337
                 plate         15         15      0.468          1      0.739        0.5
Speed: 0.6ms preprocess, 92.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to runs/detect/combined_detector2

Addestramento combinato completato. Modello salvato in: /Users/mattiagugole/Desktop/vision_project/codice/runs/detect/combined_detector2/weights/best.pt
⭐⭐⭐ RICORDA: Aggiorna il percorso del modello `model_combined_detector_path` nel file `utils/extract_data_for_strike_zone.py`
         con: '/Users/mattiagugole/Desktop/vision_project/codice/runs/detect/combined_detector2/weights/best.pt'


## 3.1 Visualizzazione Risultati Modello Unificato (Plate + Bat)

Dopo l'addestramento, questa sezione genera un video di esempio con le bounding box rilevate dal modello combinato (plate e bat) e il loro livello di confidenza. Questo serve a verificare l'accuratezza del rilevamento.

In [108]:
from ultralytics import YOLO
import cv2
import os

print("\n--- Generazione video di visualizzazione del modello combinato (Plate + Bat) ---")

# --- Configurazione per la visualizzazione ---
video_input_path = "/Users/mattiagugole/Desktop/vision_project/videos/ohtani_strike.mp4"
# Trova l'ultimo modello addestrato (come fatto nella Cella 12)
runs_detect_path = "/Users/mattiagugole/Desktop/vision_project/codice/runs/detect/"
combined_detector_base_name = "combined_detector"
created_combined_runs = [d for d in os.listdir(runs_detect_path) if os.path.isdir(os.path.join(runs_detect_path, d)) and d.startswith(combined_detector_base_name)]
last_combined_run_folder = sorted(created_combined_runs, key=lambda x: int(x.replace(combined_detector_base_name, '') or 0))[-1]

model_combined_detector_path = os.path.join(runs_detect_path, last_combined_run_folder, "weights", "best.pt")

output_video_display_path = "/Users/mattiagugole/Desktop/vision_project/videos/combined_detections_display.mp4"

# Assicurati che la cartella 'videos' esista
os.makedirs(os.path.dirname(output_video_display_path), exist_ok=True)

# --- Carica il modello addestrato ---
try:
    model_display = YOLO(model_combined_detector_path)
    print(f"Modello combinato caricato per visualizzazione da: {model_combined_detector_path}")
except Exception as e:
    print(f"Errore: Impossibile caricare il modello combinato per visualizzazione. Assicurati che il percorso sia corretto: {e}")
    exit()

# --- Prepara il video writer ---
cap = cv2.VideoCapture(video_input_path)
if not cap.isOpened():
    print(f"Errore: Impossibile aprire il video '{video_input_path}'.")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) # Mantieni lo stesso FPS del video originale

fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec per file .mp4
out_display = cv2.VideoWriter(output_video_display_path, fourcc, fps, (frame_width, frame_height))

frame_idx = 0
print(f"Inizio generazione video di visualizzazione delle detection. Salvataggio in: {output_video_display_path}")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Esegui inferenza con il modello combinato
    # verbose=False per non stampare ogni risultato della predizione nel terminale
    results = model_display.predict(source=frame, conf=0.25, verbose=False)[0] # conf=0.25 è una soglia di confidenza, puoi regolarla

    # Disegna le bounding box e il livello di confidenza
    if results.boxes is not None:
        for box in results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            
            # Recupera il nome della classe
            label = model_display.names[cls_id]

            # Scegli il colore in base alla classe
            color = (0, 255, 0) if label == 'plate' else (0, 0, 255) # Verde per plate, Blu per bat (puoi personalizzare)

            # Disegna la bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Prepara il testo per la label e la confidenza
            display_text = f"{label}: {conf:.2f}"
            
            # Posiziona il testo sopra la bounding box
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.6
            font_thickness = 2
            text_size = cv2.getTextSize(display_text, font, font_scale, font_thickness)[0]
            
            text_x = x1
            text_y = y1 - 10 if y1 - 10 > text_size[1] else y1 + text_size[1] + 10 # Posiziona sopra o sotto se troppo vicino al bordo superiore

            cv2.putText(frame, display_text, (text_x, text_y), font, font_scale, color, font_thickness, cv2.LINE_AA)

    out_display.write(frame)
    frame_idx += 1

cap.release()
out_display.release()
print(f"Video di visualizzazione detection completato e salvato in: {output_video_display_path}")
print("Controlla il video per valutare l'accuratezza del tuo modello unificato.")


--- Generazione video di visualizzazione del modello combinato (Plate + Bat) ---
Modello combinato caricato per visualizzazione da: /Users/mattiagugole/Desktop/vision_project/codice/runs/detect/combined_detector2/weights/best.pt
Inizio generazione video di visualizzazione delle detection. Salvataggio in: /Users/mattiagugole/Desktop/vision_project/videos/combined_detections_display.mp4
Video di visualizzazione detection completato e salvato in: /Users/mattiagugole/Desktop/vision_project/videos/combined_detections_display.mp4
Controlla il video per valutare l'accuratezza del tuo modello unificato.


In [103]:
import subprocess

print("\nEsecuzione di extract_data_for_strike_zone.py...")
# Assicurati che il percorso sia corretto. Il file dovrebbe essere in utils/
path_to_extract_data_script = "/Users/mattiagugole/Desktop/vision_project/utils/extract_data_for_strike_zone.py"
subprocess.run(["python", path_to_extract_data_script], check=True)
print("Estrazione dati strike zone completata. Dati salvati in utils/strike_zone_data.json.")


Esecuzione di extract_data_for_strike_zone.py...
Errore nel caricamento di uno o entrambi i modelli: [Errno 2] No such file or directory: '/Users/mattiagugole/Desktop/vision_project/codice/runs/detect/combined_detector/weights/best.pt'
Assicurati che i percorsi dei modelli siano corretti e che siano accessibili.
Per il modello combinato, verifica che il percorso in 'model_combined_detector_path' sia ESATTO.
Estrazione dati strike zone completata. Dati salvati in utils/strike_zone_data.json.


In [81]:
# Cella 17 (Code - Esegui `draw_strike_zone.py`)
import subprocess

print("\nEsecuzione di draw_strike_zone.py...")
# Assicurati che il percorso sia corretto. Questo file è nella root del tuo progetto.
path_to_draw_script = "/Users/mattiagugole/Desktop/vision_project/utils/draw_strike_zone.py"
subprocess.run(["python", path_to_draw_script], check=True)
print("Disegno della strike zone completato. Video salvato in videos/strike_zone_dynamic_output.mp4")


Esecuzione di draw_strike_zone.py...
Video output con strike zone dinamica e keypoints salvato in: /Users/mattiagugole/Desktop/vision_project/videos/strike_zone_dynamic_output.mp4
Disegno della strike zone completato. Video salvato in videos/strike_zone_dynamic_output.mp4


## 3.2 Debug: Visualizzazione della Sola Pose Estimation

Questa sezione genera un video che mostra solo le pose rilevate dal modello YOLOv8-pose (`yolov8n-pose.pt`). Questo è utile per capire se il modello rileva correttamente le persone (inclusi lanciatore, battitore, ricevitore) fin dall'inizio del video e con quale confidenza.

In [72]:
import subprocess

print("\nEsecuzione di visualize_pose_only.py per il debug della pose estimation...")
# Assicurati che il percorso sia corretto per il tuo script in utils/
path_to_pose_only_script = "/Users/mattiagugole/Desktop/vision_project/utils/visualize_pose_only.py"
subprocess.run(["python", path_to_pose_only_script], check=True)
print("Generazione video di sola pose estimation completata. Video salvato in videos/pose_only_output.mp4")


Esecuzione di visualize_pose_only.py per il debug della pose estimation...
Modello pose caricato: yolov8n-pose.pt
Inizio generazione video di sola pose estimation. Salvataggio in: /Users/mattiagugole/Desktop/vision_project/videos/pose_only_output.mp4
Video di sola pose estimation completato e salvato in: /Users/mattiagugole/Desktop/vision_project/videos/pose_only_output.mp4
Controlla questo video per vedere la performance della pose estimation in ogni frame.
Generazione video di sola pose estimation completata. Video salvato in videos/pose_only_output.mp4


## 3.3 Debug: Rilevamento Pallina 

Questa sezione esegue lo script `detect_ball.py` per tentare di rilevare la pallina da baseball usando filtri di colore, forma e movimento.
Il video di debug (`ball_detection_debug.mp4`) mostrerà le detection della pallina.

In [87]:
import subprocess

print("\nEsecuzione di detect_ball.py per il debug del rilevamento della pallina...")
path_to_ball_detection_script = "/Users/mattiagugole/Desktop/vision_project/utils/detect_ball.py"
subprocess.run(["python", path_to_ball_detection_script], check=True)
print("Generazione video di debug rilevamento pallina completata. Dati traiettoria salvati in utils/ball_trajectory.json.")


Esecuzione di detect_ball.py per il debug del rilevamento della pallina...
Inizio rilevamento pallina con MOG2. Debug video in: /Users/mattiagugole/Desktop/vision_project/videos/ball_detection_debug.mp4
Video di debug rilevamento pallina salvato in: /Users/mattiagugole/Desktop/vision_project/videos/ball_detection_debug.mp4
Traiettoria della pallina salvata in: /Users/mattiagugole/Desktop/vision_project/utils/ball_trajectory.json
Generazione video di debug rilevamento pallina completata. Dati traiettoria salvati in utils/ball_trajectory.json.
